# Phase 4C.6 — Controlled DSP augmentation (development only) — Colab launcher

Runs the locked protocol `ml/configs/dsp_augmentation_protocol.yaml` from the exact code package
(pinned snapshot manifest): recipes `visual_calibrated`, `late_fusion_original` and
`late_fusion_dsp_augmented` on the 341 development sources and the 4C.3B 5x4 folds. Features are
reused from the 4C.3B ablation caches and the 4C.4B robustness outputs (no extraction, no GPU).

* Default `MODE = "preflight"` (0 fits). `"pilot"` = fold 0 (92 fits, technical check only).
  `"full"` = 460 fits; `"analyze"` / `"verify"` need `ALLOW_FULL = True`.
* Zero locked-test access. Results are development/exploratory, never confirmatory.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

MODE = "preflight"          # "preflight" -> "pilot" -> "full" -> "analyze" -> "verify"
ALLOW_FULL = False          # must be True for MODE in {"full", "analyze", "verify"}
EXPECTED_SNAPSHOT_MANIFEST_SHA256 = "4ed15a2bf8c72d4e2b4ae5c6e6d8e2bb3afae13eb2aae0efb2ae3260eab72206"  # research/evidence/phase-4c.6a/package_receipt.json

ROOT = Path("/content/drive/MyDrive/Colab Notebooks/forensics-web-lab")
CODE_ARCHIVE = ROOT / "dsp_augmentation/inputs/phase_4c6_dsp_augmentation_code.tar.gz"
DATASET_ARCHIVE = ROOT / "phase_4c1/inputs/phase_4c1_binary_n250_reusable.tar"
ABLATION_DIR = ROOT / "visual_dsp_ablation/runs/execution_ablation"
LATE_FUSION_DIR = ROOT / "calibrated_late_fusion/runs/execution_late_fusion"
ROBUSTNESS_DIR = ROOT / "development_robustness/runs/execution_robustness"
OUTPUT_DIR = ROOT / "dsp_augmentation/runs/execution_dsp_augmentation"
REPORT_DIR = ROOT / "dsp_augmentation/report"
STAGE = Path("/content/dsp_augmentation_stage")
BUNDLE = Path("/content/development_bundle")


In [ ]:
import hashlib, json, re, shutil, tarfile
from pathlib import PurePosixPath

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def safe_extract(archive_path, destination, mode):
    with tarfile.open(archive_path, mode) as archive:
        members = archive.getmembers()
        for member in members:
            posix = PurePosixPath(member.name)
            assert member.isfile() or member.isdir(), f"Non-regular archive member: {member.name}"
            assert not posix.is_absolute() and ".." not in posix.parts and "\\" not in member.name
        archive.extractall(destination, members=members)

assert MODE in {"preflight", "pilot", "full", "analyze", "verify"}
assert MODE in {"preflight", "pilot"} or ALLOW_FULL, "full/analyze/verify requires ALLOW_FULL=True"
assert re.fullmatch(r"[0-9a-f]{64}", EXPECTED_SNAPSHOT_MANIFEST_SHA256), "EXPECTED_SNAPSHOT_MANIFEST_SHA256 must be a full SHA-256"
for path in (CODE_ARCHIVE, DATASET_ARCHIVE, ABLATION_DIR, LATE_FUSION_DIR, ROBUSTNESS_DIR):
    assert path.exists(), f"Missing input: {path}"

if STAGE.exists():
    shutil.rmtree(STAGE)
STAGE.mkdir(parents=True)
safe_extract(CODE_ARCHIVE, STAGE, "r:gz")
assert sha256_file(STAGE / "SNAPSHOT_MANIFEST.json") == EXPECTED_SNAPSHOT_MANIFEST_SHA256, "Snapshot manifest does not match the receipt"
snapshot = json.loads((STAGE / "SNAPSHOT_MANIFEST.json").read_text())
assert snapshot["experiment"] == "controlled_dsp_augmentation"
for member in snapshot["members"]:
    assert sha256_file(STAGE / member["path"]) == member["sha256"], member["path"]
print("source commit:", snapshot["source_commit"], "members:", len(snapshot["members"]))
if not (BUNDLE / "manifest_pilot_a_option_p.csv").is_file():
    BUNDLE.mkdir(parents=True, exist_ok=True)
    safe_extract(DATASET_ARCHIVE, BUNDLE, "r:")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import subprocess, sys

if MODE in {"analyze", "verify"}:
    cmd = [sys.executable, "scripts/research/analyze_dsp_augmentation.py", "--output-dir", str(OUTPUT_DIR),
           "--report-dir", str(REPORT_DIR)] + (["--verify"] if MODE == "verify" else [])
else:
    cmd = [sys.executable, "-m", "ml.training.run_dsp_augmentation", "--mode", MODE,
           "--manifest", str(BUNDLE / "manifest_pilot_a_option_p.csv"), "--data-root", str(BUNDLE),
           "--ablation-dir", str(ABLATION_DIR), "--late-fusion-dir", str(LATE_FUSION_DIR),
           "--robustness-dir", str(ROBUSTNESS_DIR), "--output-dir", str(OUTPUT_DIR)]
print("Executing:", " ".join(cmd))
subprocess.run(cmd, cwd=STAGE, check=True)
if MODE == "analyze":
    print((REPORT_DIR / "ANALYSIS_REPORT.md").read_text())
